[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/vkoul/advanced-ab-testing/blob/main/study_notes/week01_statistical_foundations.ipynb)

> **Run this notebook interactively:** Click the badge above to open in Google Colab.

# Week 1: Statistical Foundations

**Course**: Advanced A/B Testing | **Context**: QuickCart (online electronics retailer)

**Your role**: Data scientist on the experimentation platform team. Before building advanced techniques, you need rock-solid fundamentals — understanding how tests work, when they break, and how to validate them.

---

## What You'll Learn

1. Student's t-distribution and why it matters for finite samples
2. Implementing a t-test from scratch (not just calling `scipy`)
3. Three testing approaches: t-test, Mann-Whitney U, and bootstrap
4. Type I and Type II errors — measuring them empirically
5. AAB testing as a validation methodology
6. How outliers break parametric tests
7. Bootstrap confidence intervals and their limitations

---

## Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import scipy.stats as stats
import time

from collections import defaultdict
from tqdm.notebook import tqdm

plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 12
np.random.seed(42)

---

## 1. Student's t-Distribution

### The Core Idea

When comparing two groups in an A/B test, we compute a **test statistic** and ask: "How likely is this value if there's truly no difference?" To answer that, we need to know the **distribution** of the statistic under the null hypothesis.

For a sample $X_1, X_2, \ldots, X_n$ from a normal distribution $N(\mu, \sigma^2)$ with **unknown** variance, we define:

$$
\bar{X} = \frac{1}{n} \sum_{i=1}^n X_i, \quad
S^2 = \frac{1}{n-1} \sum_{i=1}^n (X_i - \bar{X})^2, \quad
t = \frac{\bar{X} - \mu}{S / \sqrt{n}}
$$

The statistic $t$ follows a **Student's t-distribution** with $(n-1)$ degrees of freedom.

### Why Not Just Use a Normal Distribution?

As $n \to \infty$, the t-distribution converges to the standard normal. But for **finite samples** (which is all we have in practice), the t-distribution has **heavier tails**. This means:

- Extreme values are more likely than the normal would predict
- Confidence intervals are wider (appropriately accounting for the uncertainty in estimating $\sigma$)
- Ignoring this leads to **underestimating** the probability of large deviations

In [ ]:
# Visualize t-distributions with different degrees of freedom
X = np.linspace(-5, 5, 200)

for k in [1, 2, 5, 20]:
    plt.plot(X, stats.t.pdf(X, k), label=f't (df={k})')

plt.plot(X, stats.norm.pdf(X), 'k--', label='Normal', linewidth=2)

plt.xlabel('x')
plt.ylabel('Density')
plt.title("Student's t-distribution vs Normal")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

**Key observation**: With only 1-2 degrees of freedom, the tails are dramatically heavier. By df=20, it's nearly indistinguishable from normal. In A/B testing, we typically have thousands of observations, so the distinction is small — but it matters for pilot tests with small samples.

---

## 2. The Two-Sample t-Test (From Scratch)

### The Problem

Given two samples — control group $X_1, \ldots, X_{n_1}$ and treatment group $Y_1, \ldots, Y_{n_2}$ — we want to test:

- $H_0$: $\mathbb{E}[X] = \mathbb{E}[Y]$ (no difference)
- $H_1$: $\mathbb{E}[X] \neq \mathbb{E}[Y]$ (some difference exists)

### Welch's t-statistic

$$
t = \frac{\bar{Y} - \bar{X}}{\sqrt{\frac{S_X^2}{n_1} + \frac{S_Y^2}{n_2}}}
$$

This statistic approximately follows $t(\nu)$ where the degrees of freedom are given by the **Welch-Satterthwaite equation**:

$$
\nu = \frac{\left(\frac{S_X^2}{n_1} + \frac{S_Y^2}{n_2}\right)^2}{\frac{S_X^4}{n_1^2(n_1-1)} + \frac{S_Y^4}{n_2^2(n_2-1)}}
$$

:::{admonition} Why Welch's test over Student's?
:class: tip
The classic Student's t-test assumes equal variances in both groups. Welch's version relaxes this assumption — it works even when variances differ. In A/B testing, treatment can change both the mean and the variance, so always prefer Welch's version.
:::

In [ ]:
def welch_degrees_of_freedom(data_a: np.ndarray, data_b: np.ndarray) -> float:
    """Compute degrees of freedom using Welch-Satterthwaite equation."""
    n_a, n_b = len(data_a), len(data_b)
    var_a = np.var(data_a, ddof=1)
    var_b = np.var(data_b, ddof=1)
    
    numerator = (var_a / n_a + var_b / n_b) ** 2
    denominator = (var_a ** 2) / (n_a ** 2 * (n_a - 1)) + (var_b ** 2) / (n_b ** 2 * (n_b - 1))
    return numerator / denominator


def t_statistic(data_a: np.ndarray, data_b: np.ndarray) -> float:
    """Compute the two-sample t-statistic (Welch's version)."""
    n_a, n_b = len(data_a), len(data_b)
    mean_a, mean_b = np.mean(data_a), np.mean(data_b)
    var_a = np.var(data_a, ddof=1)
    var_b = np.var(data_b, ddof=1)
    
    return (mean_a - mean_b) / np.sqrt(var_a / n_a + var_b / n_b)

In [ ]:
# Example: QuickCart checkout times (seconds)
np.random.seed(44)
control = np.random.normal(120, 30, size=100)  # current checkout ~120s
treatment = np.random.normal(108, 30, size=100)  # new flow ~108s (10% faster)

# Our implementation
df = welch_degrees_of_freedom(control, treatment)
t_stat = t_statistic(control, treatment)
alpha = 0.05
critical_bounds = stats.t.ppf([alpha/2, 1 - alpha/2], df=df)
p_value = 2 * stats.t.cdf(-abs(t_stat), df=df)

print(f"Degrees of freedom: {df:.1f}")
print(f"Critical region: [{critical_bounds[0]:.3f}, {critical_bounds[1]:.3f}]")
print(f"t-statistic: {t_stat:.3f}")
print(f"p-value: {p_value:.4f}")
print(f"\nSignificant at alpha=0.05? {'Yes' if p_value < alpha else 'No'}")

# Verify against scipy
scipy_result = stats.ttest_ind(control, treatment, equal_var=False)
print(f"\nScipy verification: t={scipy_result.statistic:.3f}, p={scipy_result.pvalue:.4f}")

In [ ]:
# Visualize: where does our statistic fall relative to the null distribution?
x = np.linspace(-4, 4, 500)
y = stats.t.pdf(x, df)

plt.plot(x, y, 'b-', label=f't-distribution (df={df:.0f})')

# Shade critical regions
mask_left = x < critical_bounds[0]
mask_right = x > critical_bounds[1]
plt.fill_between(x[mask_left], y[mask_left], alpha=0.3, color='red', label='Critical region (alpha=0.05)')
plt.fill_between(x[mask_right], y[mask_right], alpha=0.3, color='red')

# Mark our test statistic
plt.axvline(t_stat, color='black', linestyle='--', linewidth=2, label=f't-statistic = {t_stat:.2f}')

plt.xlabel('t')
plt.ylabel('Density')
plt.title('Two-Sample t-Test: Null Distribution and Our Statistic')
plt.legend()
plt.grid(alpha=0.3)
plt.show()

---

## 3. Three Testing Approaches

In practice, you'll encounter three main approaches to testing:

| Test | Type | Assumption | Best for |
|------|------|------------|----------|
| **t-test** | Parametric | Approximately normal (CLT) | Large samples, fast computation |
| **Mann-Whitney U** | Non-parametric | Ordinal data | Skewed data, robust to outliers |
| **Bootstrap** | Resampling | Exchangeability | Custom statistics, small samples |

Let's implement all three and understand their trade-offs.

In [ ]:
def check_ttest(a, b, alpha=0.05):
    """Student's t-test. Returns 1 if difference is significant."""
    _, pvalue = stats.ttest_ind(a, b)
    return int(pvalue < alpha)


def check_mannwhitney(a, b, alpha=0.05):
    """Mann-Whitney U test. Returns 1 if difference is significant."""
    _, pvalue = stats.mannwhitneyu(a, b, alternative='two-sided')
    return int(pvalue < alpha)


def check_bootstrap(a, b, func=np.mean, n_bootstrap=1000, alpha=0.05):
    """Bootstrap test. Returns 1 if CI for difference excludes zero."""
    a_samples = np.random.choice(a, size=(len(a), n_bootstrap))
    b_samples = np.random.choice(b, size=(len(b), n_bootstrap))
    differences = func(a_samples, axis=0) - func(b_samples, axis=0)
    ci_low = np.quantile(differences, alpha / 2)
    ci_high = np.quantile(differences, 1 - alpha / 2)
    return int(ci_low > 0 or ci_high < 0)


tests = {
    'ttest': check_ttest,
    'mann_whitney': check_mannwhitney,
    'bootstrap': check_bootstrap
}

### Sanity Check: No Effect (A/A Test)

When both groups come from the **same distribution**, a well-calibrated test should reject the null only ~5% of the time (matching our alpha).

In [ ]:
results = defaultdict(list)

for _ in tqdm(range(1000)):
    a = np.random.normal(0, 1, 100)
    b = np.random.normal(0, 1, 100)  # same distribution!
    for name, test in tests.items():
        results[name].append(test(a, b))

print("False positive rates (should be ~0.05):")
for name, values in results.items():
    print(f"  {name:15s}: {np.mean(values):.3f}")

### With a Real Effect

Now let's add a true difference (shift of 0.4 standard deviations) and see how often each test detects it.

In [ ]:
results = defaultdict(list)

for _ in tqdm(range(1000)):
    a = np.random.normal(0, 1, 100)
    b = np.random.normal(0.4, 1, 100)  # shifted by 0.4
    for name, test in tests.items():
        results[name].append(test(a, b))

print("Detection rates (power) with effect=0.4:")
for name, values in results.items():
    print(f"  {name:15s}: {np.mean(values):.3f}")

### Computation Time Comparison

An important practical consideration: how fast are these tests as sample size grows?

In [ ]:
sample_sizes = np.logspace(2, 5, 15).astype(int)
timings = {name: [] for name in tests}

for n in tqdm(sample_sizes):
    a = np.random.normal(0, 1, n)
    b = np.random.normal(0, 1, n)
    for name, test in tests.items():
        t0 = time.time()
        test(a, b)
        timings[name].append(time.time() - t0)

for name, times in timings.items():
    plt.plot(sample_sizes, times, 'o-', label=name)

plt.xlabel('Sample size')
plt.ylabel('Time (seconds)')
plt.title('Computation Time by Sample Size')
plt.xscale('log')
plt.yscale('log')
plt.legend()
plt.grid(alpha=0.3)
plt.show()

**Takeaway**: The t-test is orders of magnitude faster than bootstrap at large sample sizes. Mann-Whitney scales moderately. For QuickCart's scale (millions of users), bootstrap is impractical as a primary test — use it for confidence intervals on custom statistics, not as your main hypothesis test.

---

## 4. Type I and Type II Errors

### Definitions

| Error | Name | What happens | Consequence at QuickCart |
|-------|------|--------------|-------------------------|
| **Type I** | False Positive | Reject $H_0$ when it's true | Launch a feature that does nothing (or hurts) |
| **Type II** | False Negative | Fail to reject $H_0$ when $H_1$ is true | Miss a real improvement, leave money on the table |

- **Type I error rate** = $\alpha$ (typically 0.05)
- **Type II error rate** = $\beta$ (typically 0.20)
- **Power** = $1 - \beta$ (probability of detecting a real effect)

### AAB Testing: Validating Your Tests

Before trusting a test in production, validate it with **AAB testing**:

1. Take **two samples from control** (A1, A2) and **one from treatment** (B)
2. Run your test on A1 vs A2 — it should rarely reject (measures Type I error)
3. Run your test on A1 vs B — it should often reject (measures power / 1 - Type II error)

This gives you empirical confidence that your testing pipeline is correctly calibrated.

In [ ]:
def run_aab_experiment(dist_a, dist_b, sample_sizes, n_simulations=1000):
    """
    Run AAB experiment: measure Type I and Type II errors across sample sizes.
    
    dist_a: scipy distribution for control
    dist_b: scipy distribution for treatment
    """
    # Only use t-test and mann-whitney (bootstrap too slow for this)
    fast_tests = {'ttest': check_ttest, 'mann_whitney': check_mannwhitney}
    
    type1_errors = defaultdict(list)
    type2_errors = defaultdict(list)
    
    for n in tqdm(sample_sizes):
        aa_results = defaultdict(list)
        ab_results = defaultdict(list)
        
        for _ in range(n_simulations):
            a1 = dist_a.rvs(size=n)
            a2 = dist_a.rvs(size=n)
            b = dist_b.rvs(size=n)
            
            for name, test in fast_tests.items():
                aa_results[name].append(test(a1, a2))  # should NOT reject
                ab_results[name].append(test(a1, b))   # should reject
        
        for name in fast_tests:
            type1_errors[name].append(np.mean(aa_results[name]))
            type2_errors[name].append(1 - np.mean(ab_results[name]))
    
    return type1_errors, type2_errors

### Experiment 1: Two Shifted Normal Distributions

Control ~ N(0, 1), Treatment ~ N(0.2, 1). A small but real effect.

In [ ]:
sample_sizes = np.logspace(2, 3, 10).astype(int)

type1, type2 = run_aab_experiment(
    dist_a=stats.norm(loc=0, scale=1),
    dist_b=stats.norm(loc=0.2, scale=1),
    sample_sizes=sample_sizes
)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for name, errors in type1.items():
    axes[0].plot(sample_sizes, errors, 'o-', label=name)
axes[0].axhline(0.05, color='red', linestyle='--', label='alpha=0.05')
axes[0].set_title('Type I Error Rate (should stay at 0.05)')
axes[0].set_xlabel('Sample size')
axes[0].set_ylim([-0.02, 0.15])
axes[0].legend()
axes[0].grid(alpha=0.3)

for name, errors in type2.items():
    axes[1].plot(sample_sizes, errors, 'o-', label=name)
axes[1].set_title('Type II Error Rate (should decrease with n)')
axes[1].set_xlabel('Sample size')
axes[1].set_ylim([-0.02, 1.05])
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

**Observations**:
- Type I error stays at ~0.05 regardless of sample size (both tests are correctly calibrated)
- Type II error decreases as sample size grows (more data = more power)
- Both tests perform similarly on normal data; t-test may have a slight edge

### Experiment 2: Exponential Distributions (Skewed Data)

Revenue data is often right-skewed. Let's see how the tests behave with exponential distributions.

In [ ]:
type1_exp, type2_exp = run_aab_experiment(
    dist_a=stats.expon(loc=0, scale=1.0),
    dist_b=stats.expon(loc=0, scale=1.2),
    sample_sizes=sample_sizes
)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for name, errors in type1_exp.items():
    axes[0].plot(sample_sizes, errors, 'o-', label=name)
axes[0].axhline(0.05, color='red', linestyle='--', label='alpha=0.05')
axes[0].set_title('Type I Error (Exponential data)')
axes[0].set_xlabel('Sample size')
axes[0].set_ylim([-0.02, 0.15])
axes[0].legend()
axes[0].grid(alpha=0.3)

for name, errors in type2_exp.items():
    axes[1].plot(sample_sizes, errors, 'o-', label=name)
axes[1].set_title('Type II Error (Exponential data)')
axes[1].set_xlabel('Sample size')
axes[1].set_ylim([-0.02, 1.05])
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

---

## 5. Sensitivity to Outliers

### Why This Matters at QuickCart

E-commerce revenue data is notorious for outliers: a single B2B customer buying 500 laptops can dominate the mean of a group with 1,000 regular customers. How do our tests handle this?

### Experiment: Injecting Outliers

We fix the sample size and progressively inject larger outliers into one group, then measure how each test is affected.

In [ ]:
# Only t-test and Mann-Whitney (bootstrap too slow for this many iterations)
fast_tests = {'ttest': check_ttest, 'mann_whitney': check_mannwhitney}

outlier_values = np.arange(-150, 301, 15)
type1_outlier = defaultdict(list)
type2_outlier = defaultdict(list)
sample_size = 500
n_simulations = 1000

for outlier in tqdm(outlier_values):
    t1_results = defaultdict(list)
    t2_results = defaultdict(list)
    
    for _ in range(n_simulations):
        a1 = np.random.normal(0, 1, sample_size)
        a1[0] = outlier  # inject single outlier
        a2 = np.random.normal(0, 1, sample_size)
        b = np.random.normal(0.2, 1, sample_size)
        
        for name, test in fast_tests.items():
            t1_results[name].append(test(a1, a2))
            t2_results[name].append(test(a1, b))
    
    for name in fast_tests:
        type1_outlier[name].append(np.mean(t1_results[name]))
        type2_outlier[name].append(1 - np.mean(t2_results[name]))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for name in fast_tests:
    axes[0].plot(outlier_values, type1_outlier[name], label=name)
axes[0].axhline(0.05, color='red', linestyle='--', alpha=0.5)
axes[0].set_title('Type I Error vs Outlier Magnitude')
axes[0].set_xlabel('Outlier value')
axes[0].set_ylim([-0.02, 0.3])
axes[0].legend()
axes[0].grid(alpha=0.3)

for name in fast_tests:
    axes[1].plot(outlier_values, type2_outlier[name], label=name)
axes[1].set_title('Type II Error vs Outlier Magnitude')
axes[1].set_xlabel('Outlier value')
axes[1].set_ylim([-0.02, 1.05])
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

### What's Happening

- **t-test**: As the outlier grows, the Type II error rate *skyrockets*. The outlier inflates the variance estimate $S^2$, which makes the denominator of the t-statistic large, making it harder to reject $H_0$. The test loses power.

- **Mann-Whitney**: Almost unaffected by the outlier. It uses ranks (ordinal position), so a single extreme value only affects comparisons involving that one observation.

:::{admonition} Practical Implication
:class: warning
If your metric has heavy tails or potential outliers (revenue, session duration, page load time), either:
1. Use Mann-Whitney as a robustness check
2. Cap/winsorize outliers before applying the t-test
3. Use log-transform for multiplicative metrics

We'll cover variance reduction techniques (including outlier handling) in Weeks 3-5.
:::

---

## 6. Bootstrap Confidence Intervals

### The Bootstrap Idea

The bootstrap lets us estimate the **sampling distribution** of any statistic without knowing the underlying distribution. The procedure:

1. Draw $B$ resamples (with replacement) from your data, each of size $n$
2. Compute your statistic on each resample
3. The distribution of these $B$ estimates approximates the sampling distribution
4. Use percentiles for confidence intervals

### Example: Skewness of Revenue Data

QuickCart's per-order revenue is right-skewed (most orders are small, a few are large). Let's estimate the skewness and build a bootstrap CI.

In [ ]:
# Simulate revenue-like data (exponential distribution)
distribution = stats.expon(loc=0, scale=50)  # mean $50 per order
true_skewness = float(distribution.stats(moments='s'))  # theoretical: 2.0 for exponential

np.random.seed(23)
sample = distribution.rvs(size=100)

print(f"True skewness of exponential distribution: {true_skewness:.2f}")
print(f"Sample skewness (point estimate): {stats.skew(sample):.3f}")
print(f"Sample size: {len(sample)}")

In [ ]:
# Bootstrap confidence intervals for skewness at different subsample sizes
B = 10000
subsample_sizes = np.arange(20, 101, 5)
bootstrap_distributions = []

for size in tqdm(subsample_sizes):
    subsample = sample[:size]
    resamples = np.random.choice(subsample, size=(size, B))
    bootstrap_skewness = stats.skew(resamples, axis=0)
    bootstrap_distributions.append(bootstrap_skewness)

alpha = 0.05
distributions_array = np.array(bootstrap_distributions)
ci_bounds = np.quantile(distributions_array, [alpha/2, 1-alpha/2], axis=1)
means = np.mean(distributions_array, axis=1)

In [ ]:
plt.fill_between(subsample_sizes, ci_bounds[0], ci_bounds[1], alpha=0.2, label='95% Bootstrap CI')
plt.plot(subsample_sizes, means, 'b-', label='Bootstrap mean')
plt.axhline(true_skewness, color='red', linestyle='--', linewidth=2, label=f'True skewness = {true_skewness}')
plt.plot(subsample_sizes, [stats.skew(sample[:n]) for n in subsample_sizes], 'g:', label='Point estimate')

plt.xlabel('Sample size')
plt.ylabel('Skewness')
plt.title('Bootstrap CI for Skewness vs Sample Size')
plt.legend()
plt.grid(alpha=0.3)
plt.show()

### The Limitation: Bootstrap Can't Overcome Bad Samples

Notice that the true skewness (2.0) may fall **outside** the bootstrap CI. This happens because:

1. Bootstrap resamples from the **empirical distribution** (your data)
2. If your sample doesn't represent the tail behavior of the true distribution, bootstrap can't recover that information
3. The CI reflects uncertainty about the sample, not about the population

:::{admonition} When Bootstrap Works Well
:class: note
- The statistic is a smooth function of the data (means, quantiles)
- The sample is representative of the population
- Sample size is "large enough" for the complexity of the statistic

When it struggles:
- Extreme quantiles or tail-dependent statistics
- Very small samples from heavy-tailed distributions
- Statistics that depend on the maximum or minimum
:::

In [ ]:
# Demonstrate: with access to the true distribution, we can see how often
# the bootstrap CI actually covers the true value

# Generate many independent samples and check coverage
n_experiments = 1000
sample_size = 100
coverage_count = 0

for _ in range(n_experiments):
    data = distribution.rvs(size=sample_size)
    resamples = np.random.choice(data, size=(sample_size, 1000))
    boot_stats = stats.skew(resamples, axis=0)
    ci_lo, ci_hi = np.quantile(boot_stats, [0.025, 0.975])
    if ci_lo <= true_skewness <= ci_hi:
        coverage_count += 1

print(f"Bootstrap 95% CI coverage for skewness (n={sample_size}): {coverage_count/n_experiments:.1%}")
print(f"Expected: 95%")
print(f"\nThe undercoverage shows bootstrap struggles with tail-heavy statistics at moderate n.")

---

## 7. P-Value Distribution Under the Null

A correctly calibrated test produces **uniformly distributed p-values** when the null hypothesis is true. This is a powerful diagnostic.

If your p-values are NOT uniform under the null, something is wrong with your test (violated assumptions, data leakage, etc.).

In [ ]:
# Well-calibrated: normal data with t-test
pvals_normal = []
for _ in range(5000):
    a = np.random.normal(0, 1, 100)
    b = np.random.normal(0, 1, 100)
    _, p = stats.ttest_ind(a, b)
    pvals_normal.append(p)

# Poorly calibrated: log-normal data with t-test (small samples)
pvals_lognormal = []
for _ in range(5000):
    a = np.random.lognormal(-3, 3, 30)
    b = np.random.lognormal(-3, 3, 30)
    _, p = stats.ttest_ind(a, b)
    pvals_lognormal.append(p)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(pvals_normal, bins=50, density=True, alpha=0.7)
axes[0].axhline(1, color='red', linestyle='--', label='Uniform (expected)')
axes[0].set_title('P-values under null: Normal data (well-calibrated)')
axes[0].set_xlabel('p-value')
axes[0].legend()

axes[1].hist(pvals_lognormal, bins=50, density=True, alpha=0.7, color='orange')
axes[1].axhline(1, color='red', linestyle='--', label='Uniform (expected)')
axes[1].set_title('P-values under null: Log-normal data, n=30 (miscalibrated)')
axes[1].set_xlabel('p-value')
axes[1].legend()

plt.tight_layout()
plt.show()

print(f"False positive rate (normal, n=100):     {np.mean(np.array(pvals_normal) < 0.05):.3f}")
print(f"False positive rate (log-normal, n=30):  {np.mean(np.array(pvals_lognormal) < 0.05):.3f}")

**Key insight**: With extremely heavy-tailed data (log-normal with high variance) and small samples, the t-test becomes miscalibrated. The p-values pile up near 0, leading to an inflated false positive rate. This is why **always validate your tests on A/A data** before trusting them on A/B comparisons.

---

## Summary

| Concept | Key Takeaway |
|---------|-------------|
| t-distribution | Use it (not normal) for finite samples; heavier tails account for variance uncertainty |
| t-test | Fast, powerful for normal-ish data; breaks with outliers |
| Mann-Whitney | Robust to outliers (rank-based); slightly less powerful on clean normal data |
| Bootstrap | Flexible (any statistic); slow at scale; can't overcome bad samples |
| Type I error | Should be ~alpha; validate with A/A tests |
| Type II error | Decreases with sample size; this is why power analysis matters |
| AAB testing | Gold standard for validating your testing pipeline |
| P-value distribution | Should be uniform under null; non-uniformity = broken assumptions |

### What's Next

In **Week 2**, we'll apply these foundations to real-world QuickCart data — handling messy metrics, building synthetic metrics with ML, and analyzing actual A/B test results where things don't go as cleanly as they did here.